# Aoranema — TMDB Metadata Collection

Notebook tahap 2 setelah baseline recommendation V1.

Tujuan:

```text
MovieLens 32M
→ links.csv (tmdbId)
→ TMDB API
→ metadata film lebih kaya
→ tmdb_metadata.parquet / .csv
```

Metadata yang dikumpulkan:
- release date / year
- runtime
- original language
- genres
- collection / franchise
- production companies
- director
- writers
- top 5 cast
- keywords

## Sebelum Run All di Kaggle

1. Tambahkan private dataset `aoranema-movielens-32m` sebagai Input.
2. Aktifkan Internet pada notebook.
3. Buka **Add-ons → Secrets**.
4. Buat secret:
   - Label: `TMDB_TOKEN`
   - Value: paste API Read Access Token TMDB
5. Pastikan secret tersebut diaktifkan/attached ke notebook.
6. Jangan pernah menulis token asli langsung di cell.

Notebook ini tidak mencetak token ke output.

## Strategi

Satu request per film memakai movie-details + `append_to_response=credits,keywords`.

Notebook memiliki:
- test token
- test 5 film
- retry
- handling HTTP 429
- checkpoint
- resume
- failure log
- Parquet + CSV output

Kita sengaja tidak mengambil current TMDB `popularity`, `vote_average`, dan `vote_count` sebagai feature historis karena nilainya adalah kondisi saat API dipanggil sekarang dan berisiko menyebabkan temporal leakage.

In [1]:
from pathlib import Path
import json
import random
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SECRET_LABEL = "TMDB_TOKEN"

TMDB_BASE_URL = "https://api.themoviedb.org/3"
LANGUAGE = "en-US"

REQUESTS_PER_SECOND = 8.0
REQUEST_INTERVAL = 1.0 / REQUESTS_PER_SECOND
REQUEST_TIMEOUT = 30
MAX_RETRIES = 6

TOP_CAST_N = 5
MAX_WRITERS = 10

# Pertama jalankan True untuk mengetes 5 film.
# Setelah berhasil, ubah ke False lalu Run All.
TEST_MODE = False
TEST_N_MOVIES = 5

# None = seluruh film yang belum diproses.
MAX_MOVIES_PER_RUN = None

CHECKPOINT_EVERY = 500

OUTPUT_DIR = Path("/kaggle/working/aoranema_tmdb_metadata")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "tmdb_metadata_checkpoint.parquet"
FAILED_PATH = OUTPUT_DIR / "failed_tmdb_requests.csv"
FINAL_PARQUET_PATH = OUTPUT_DIR / "tmdb_metadata.parquet"
FINAL_CSV_PATH = OUTPUT_DIR / "tmdb_metadata.csv"
SUMMARY_PATH = OUTPUT_DIR / "metadata_info.json"

print("TEST_MODE:", TEST_MODE)
print("Output:", OUTPUT_DIR)
print("Request rate:", REQUESTS_PER_SECOND, "req/s")

TEST_MODE: False
Output: /kaggle/working/aoranema_tmdb_metadata
Request rate: 8.0 req/s


In [2]:
# 1. Load token dari Kaggle Secret

try:
    from kaggle_secrets import UserSecretsClient
except ImportError as e:
    raise RuntimeError(
        "kaggle_secrets hanya tersedia di Kaggle Notebook."
    ) from e

user_secrets = UserSecretsClient()

try:
    TMDB_TOKEN = user_secrets.get_secret(SECRET_LABEL)
except Exception as e:
    raise RuntimeError(
        "Secret TMDB_TOKEN tidak dapat dibaca. "
        "Buka Add-ons → Secrets, buat label TMDB_TOKEN, "
        "paste API Read Access Token, lalu attach secret ke notebook."
    ) from e

assert TMDB_TOKEN and len(TMDB_TOKEN.strip()) > 20

print("TMDB_TOKEN berhasil dibaca dari Kaggle Secret.")
print("Token tidak ditampilkan.")

TMDB_TOKEN berhasil dibaca dari Kaggle Secret.
Token tidak ditampilkan.


In [3]:
# 2. Test koneksi TMDB

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json",
})

response = session.get(
    f"{TMDB_BASE_URL}/configuration",
    timeout=REQUEST_TIMEOUT,
)

print("HTTP status:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(
        f"TMDB connection gagal: HTTP {response.status_code} "
        f"{response.text[:300]}"
    )

print("TMDB connection: OK")

HTTP status: 200
TMDB connection: OK


In [4]:
# 3. Auto-detect MovieLens 32M

KAGGLE_INPUT = Path("/kaggle/input")
assert KAGGLE_INPUT.exists()

links_candidates = list(KAGGLE_INPUT.rglob("links.csv"))

print("links.csv candidates:")
for p in links_candidates:
    print(" -", p)

DATA_DIR = None
for links_path in links_candidates:
    candidate = links_path.parent
    if (
        (candidate / "movies.csv").exists()
        and (candidate / "ratings.csv").exists()
    ):
        DATA_DIR = candidate
        break

assert DATA_DIR is not None, (
    "MovieLens 32M tidak ditemukan. "
    "Tambahkan aoranema-movielens-32m sebagai Input."
)

LINKS_PATH = DATA_DIR / "links.csv"
MOVIES_PATH = DATA_DIR / "movies.csv"

print("\nMovieLens dir:", DATA_DIR)
print("links.csv :", LINKS_PATH)
print("movies.csv:", MOVIES_PATH)

links.csv candidates:
 - /kaggle/input/datasets/onallaaldeanuva/aoranema-movielens-32m/ml-32m/links.csv

MovieLens dir: /kaggle/input/datasets/onallaaldeanuva/aoranema-movielens-32m/ml-32m
links.csv : /kaggle/input/datasets/onallaaldeanuva/aoranema-movielens-32m/ml-32m/links.csv
movies.csv: /kaggle/input/datasets/onallaaldeanuva/aoranema-movielens-32m/ml-32m/movies.csv


In [5]:
# 4. Load mapping MovieLens → TMDB

links = pd.read_csv(
    LINKS_PATH,
    dtype={
        "movieId": "int32",
        "imdbId": "Int64",
        "tmdbId": "Int64",
    },
)

movies = pd.read_csv(
    MOVIES_PATH,
    dtype={
        "movieId": "int32",
        "title": "string",
        "genres": "string",
    },
)

movie_map = links.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left",
)

total_movies = len(movie_map)
missing_tmdb = int(movie_map["tmdbId"].isna().sum())

movie_map_valid = (
    movie_map.dropna(subset=["tmdbId"])
    .copy()
    .sort_values("movieId")
    .reset_index(drop=True)
)

movie_map_valid["tmdbId"] = movie_map_valid["tmdbId"].astype("int64")

print("MovieLens movies     :", f"{total_movies:,}")
print("Missing tmdbId       :", f"{missing_tmdb:,}")
print("Valid tmdbId mappings:", f"{len(movie_map_valid):,}")

display(movie_map_valid.head())

MovieLens movies     : 87,585
Missing tmdbId       : 124
Valid tmdbId mappings: 87,461


,movieId,imdbId,tmdbId,title,genres
0,1,114709,862,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,113497,8844,Jumanji (1995),Adventure|Children|Fantasy
2,3,113228,15602,Grumpier Old Men (1995),Comedy|Romance
3,4,114885,31357,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,113041,11862,Father of the Bride Part II (1995),Comedy


## Resume / checkpoint

Notebook mencoba melanjutkan dari:
1. `/kaggle/working/.../tmdb_metadata_checkpoint.parquet`, atau
2. checkpoint/final metadata yang kamu tambahkan kembali sebagai Kaggle Input.

Jadi kalau proses harus dibagi beberapa sesi, simpan output sesi sebelumnya lalu tambahkan kembali sebagai Input.

In [6]:
# 5. Resume checkpoint

def locate_resume_checkpoint():
    if CHECKPOINT_PATH.exists():
        return CHECKPOINT_PATH

    candidates = list(
        Path("/kaggle/input").rglob("tmdb_metadata_checkpoint.parquet")
    )
    if candidates:
        print("Checkpoint input ditemukan:")
        for p in candidates:
            print(" -", p)
        return candidates[0]

    candidates = list(
        Path("/kaggle/input").rglob("tmdb_metadata.parquet")
    )
    if candidates:
        print("Final metadata input ditemukan:")
        for p in candidates:
            print(" -", p)
        return candidates[0]

    return None

RESUME_PATH = locate_resume_checkpoint()

if RESUME_PATH is not None:
    existing_df = pd.read_parquet(RESUME_PATH)
    existing_records = existing_df.to_dict(orient="records")
    processed_tmdb_ids = set(
        pd.to_numeric(existing_df["tmdbId"], errors="coerce")
        .dropna()
        .astype("int64")
        .tolist()
    )
    print(
        f"Resume: {len(existing_records):,} rows dari {RESUME_PATH}"
    )
else:
    existing_records = []
    processed_tmdb_ids = set()
    print("Belum ada checkpoint.")

Belum ada checkpoint.


In [7]:
# 6. Helper ekstraksi metadata

WRITING_JOBS = {
    "Writer",
    "Screenplay",
    "Story",
    "Teleplay",
    "Novel",
    "Characters",
    "Adaptation",
}

def unique_people(items, max_items=None):
    seen = set()
    result = []

    for item in items:
        pid = item.get("id")
        if pid is None or pid in seen:
            continue

        seen.add(pid)
        result.append({
            "id": int(pid),
            "name": item.get("name"),
        })

        if max_items is not None and len(result) >= max_items:
            break

    return result

def ids_json(items):
    return json.dumps(
        [x["id"] for x in items if x.get("id") is not None],
        ensure_ascii=False,
    )

def names_json(items):
    return json.dumps(
        [x["name"] for x in items if x.get("name")],
        ensure_ascii=False,
    )

def extract_tmdb_metadata(ml_row, data):
    credits = data.get("credits") or {}
    crew = credits.get("crew") or []
    cast = credits.get("cast") or []

    directors = unique_people([
        x for x in crew if x.get("job") == "Director"
    ])

    writers = unique_people(
        [
            x for x in crew
            if (
                x.get("department") == "Writing"
                or x.get("job") in WRITING_JOBS
            )
        ],
        max_items=MAX_WRITERS,
    )

    cast_sorted = sorted(
        cast,
        key=lambda x: x.get("order", 999999),
    )
    top_cast = unique_people(
        cast_sorted,
        max_items=TOP_CAST_N,
    )

    keyword_block = data.get("keywords") or {}
    keywords = [
        {"id": int(x["id"]), "name": x.get("name")}
        for x in (keyword_block.get("keywords") or [])
        if x.get("id") is not None
    ]

    genres = [
        {"id": int(x["id"]), "name": x.get("name")}
        for x in (data.get("genres") or [])
        if x.get("id") is not None
    ]

    companies = [
        {"id": int(x["id"]), "name": x.get("name")}
        for x in (data.get("production_companies") or [])
        if x.get("id") is not None
    ]

    collection = data.get("belongs_to_collection")
    collection_id = (
        int(collection["id"])
        if isinstance(collection, dict)
        and collection.get("id") is not None
        else None
    )
    collection_name = (
        collection.get("name")
        if isinstance(collection, dict)
        else None
    )

    release_date = data.get("release_date") or None
    try:
        release_year = int(release_date[:4]) if release_date else None
    except Exception:
        release_year = None

    imdb_id = ml_row.get("imdbId")
    imdb_id = int(imdb_id) if pd.notna(imdb_id) else None

    return {
        "movieId": int(ml_row["movieId"]),
        "tmdbId": int(ml_row["tmdbId"]),
        "imdbId": imdb_id,

        "movielens_title": ml_row.get("title"),
        "movielens_genres": ml_row.get("genres"),

        "tmdb_title": data.get("title"),
        "original_title": data.get("original_title"),
        "release_date": release_date,
        "release_year": release_year,
        "runtime": data.get("runtime"),
        "original_language": data.get("original_language"),

        "genre_ids_json": ids_json(genres),
        "genre_names_json": names_json(genres),

        "director_ids_json": ids_json(directors),
        "director_names_json": names_json(directors),

        "writer_ids_json": ids_json(writers),
        "writer_names_json": names_json(writers),

        "top_cast_ids_json": ids_json(top_cast),
        "top_cast_names_json": names_json(top_cast),

        "keyword_ids_json": ids_json(keywords),
        "keyword_names_json": names_json(keywords),

        "production_company_ids_json": ids_json(companies),
        "production_company_names_json": names_json(companies),

        "collection_id": collection_id,
        "collection_name": collection_name,

        "n_directors": len(directors),
        "n_writers": len(writers),
        "n_top_cast": len(top_cast),
        "n_keywords": len(keywords),
        "n_genres": len(genres),
    }

In [8]:
# 7. Request TMDB dengan retry dan rate-limit handling

def fetch_tmdb_movie(tmdb_id):
    url = f"{TMDB_BASE_URL}/movie/{int(tmdb_id)}"

    params = {
        "language": LANGUAGE,
        "append_to_response": "credits,keywords",
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code == 200:
                return response.json(), None

            if response.status_code == 404:
                return None, "404_not_found"

            if response.status_code == 401:
                raise RuntimeError(
                    "HTTP 401. Periksa API Read Access Token."
                )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")
                wait = (
                    float(retry_after)
                    if retry_after
                    else min(2 ** attempt, 30) + random.random()
                )
                print(
                    f"\n429 tmdbId={tmdb_id}; tunggu {wait:.1f}s"
                )
                time.sleep(wait)
                continue

            if 500 <= response.status_code < 600:
                wait = min(2 ** attempt, 30) + random.random()
                time.sleep(wait)
                continue

            return None, f"http_{response.status_code}"

        except requests.RequestException as e:
            if attempt == MAX_RETRIES - 1:
                return None, f"request_error:{type(e).__name__}"

            wait = min(2 ** attempt, 30) + random.random()
            time.sleep(wait)

    return None, "max_retries_exceeded"

In [9]:
# 8. Test 5 film terlebih dahulu

test_rows = movie_map_valid.head(TEST_N_MOVIES)
test_output = []

for _, row in test_rows.iterrows():
    data, error = fetch_tmdb_movie(row["tmdbId"])

    if error is not None:
        print(
            f"FAILED movieId={row['movieId']} "
            f"tmdbId={row['tmdbId']}: {error}"
        )
        continue

    record = extract_tmdb_metadata(row, data)
    test_output.append(record)

    print(
        f"OK | movieId={record['movieId']} | "
        f"tmdbId={record['tmdbId']} | "
        f"{record['tmdb_title']}"
    )

    time.sleep(REQUEST_INTERVAL)

test_df = pd.DataFrame(test_output)

assert len(test_df) > 0, (
    "Tidak ada test movie yang berhasil."
)

display(
    test_df[
        [
            "movieId",
            "tmdbId",
            "tmdb_title",
            "release_year",
            "director_names_json",
            "top_cast_names_json",
            "keyword_names_json",
        ]
    ]
)

print("\nTMDB metadata test: BERHASIL")

OK | movieId=1 | tmdbId=862 | Toy Story
OK | movieId=2 | tmdbId=8844 | Jumanji
OK | movieId=3 | tmdbId=15602 | Grumpier Old Men
OK | movieId=4 | tmdbId=31357 | Waiting to Exhale
OK | movieId=5 | tmdbId=11862 | Father of the Bride Part II


,movieId,tmdbId,tmdb_title,release_year,director_names_json,top_cast_names_json,keyword_names_json
0,1,862,Toy Story,1995,"[""John Lasseter""]","[""Tom Hanks"", ""Tim Allen"", ""Don Rickles"", ""Jim...","[""rescue"", ""friendship"", ""mission"", ""jealousy""..."
1,2,8844,Jumanji,1995,"[""Joe Johnston""]","[""Robin Williams"", ""Kirsten Dunst"", ""Bradley P...","[""based on novel or book"", ""giant insect"", ""bo..."
2,3,15602,Grumpier Old Men,1995,"[""Howard Deutch""]","[""Walter Matthau"", ""Jack Lemmon"", ""Ann-Margret...","[""fishing"", ""sequel"", ""old man"", ""best friend""..."
3,4,31357,Waiting to Exhale,1995,"[""Forest Whitaker""]","[""Whitney Houston"", ""Angela Bassett"", ""Loretta...","[""based on novel or book"", ""single mother"", ""d..."
4,5,11862,Father of the Bride Part II,1995,"[""Charles Shyer""]","[""Steve Martin"", ""Diane Keaton"", ""Martin Short...","[""daughter"", ""baby"", ""parent child relationshi..."



TMDB metadata test: BERHASIL


## Setelah test berhasil

Kembali ke Cell 0 lalu ubah:

```python
TEST_MODE = False
```

Kemudian **Run All** lagi untuk full collection.

Dengan `TEST_MODE=True`, cell full collection sengaja dilewati.

In [10]:
# 9. Checkpoint helper

def load_failed_records():
    if FAILED_PATH.exists():
        return pd.read_csv(FAILED_PATH).to_dict(orient="records")

    candidates = list(
        Path("/kaggle/input").rglob("failed_tmdb_requests.csv")
    )
    if candidates:
        return pd.read_csv(candidates[0]).to_dict(orient="records")

    return []

def save_checkpoint(records, failures):
    if records:
        checkpoint_df = (
            pd.DataFrame(records)
            .drop_duplicates(subset=["tmdbId"], keep="last")
            .sort_values("movieId")
            .reset_index(drop=True)
        )
        checkpoint_df.to_parquet(
            CHECKPOINT_PATH,
            index=False,
        )

    if failures:
        failed_df = (
            pd.DataFrame(failures)
            .drop_duplicates(subset=["tmdbId"], keep="last")
            .sort_values("movieId")
            .reset_index(drop=True)
        )
        failed_df.to_csv(
            FAILED_PATH,
            index=False,
        )

In [11]:
# 10. Full collection

records = list(existing_records)
failures = load_failed_records()

permanent_404_ids = {
    int(x["tmdbId"])
    for x in failures
    if x.get("error") == "404_not_found"
    and x.get("tmdbId") is not None
}

remaining = movie_map_valid[
    ~movie_map_valid["tmdbId"].isin(processed_tmdb_ids)
].copy()

remaining = remaining[
    ~remaining["tmdbId"].isin(permanent_404_ids)
].reset_index(drop=True)

if MAX_MOVIES_PER_RUN is not None:
    remaining = remaining.head(MAX_MOVIES_PER_RUN)

print("Already collected :", f"{len(processed_tmdb_ids):,}")
print("Permanent 404     :", f"{len(permanent_404_ids):,}")
print("Remaining this run:", f"{len(remaining):,}")

estimated_minutes = len(remaining) * REQUEST_INTERVAL / 60
print(
    "Minimum request-time estimate:",
    f"{estimated_minutes:.1f} minutes",
    "(di luar overhead/retry)"
)

if TEST_MODE:
    print(
        "\nTEST_MODE=True → full collection dilewati.\n"
        "Jika test berhasil, ubah TEST_MODE=False lalu Run All."
    )
else:
    last_request_started = 0.0

    for i, (_, row) in enumerate(
        tqdm(
            remaining.iterrows(),
            total=len(remaining),
            desc="Collecting TMDB metadata",
        ),
        start=1,
    ):
        elapsed = time.time() - last_request_started
        if elapsed < REQUEST_INTERVAL:
            time.sleep(REQUEST_INTERVAL - elapsed)

        last_request_started = time.time()

        data, error = fetch_tmdb_movie(row["tmdbId"])

        if error is not None:
            failures.append({
                "movieId": int(row["movieId"]),
                "tmdbId": int(row["tmdbId"]),
                "error": error,
            })
        else:
            records.append(
                extract_tmdb_metadata(row, data)
            )

        if (
            i % CHECKPOINT_EVERY == 0
            or i == len(remaining)
        ):
            save_checkpoint(records, failures)
            print(
                f"\nCheckpoint saved | "
                f"processed this run={i:,} | "
                f"records={len(records):,} | "
                f"failures={len(failures):,}"
            )

    print("\nFull collection loop selesai.")

Already collected : 0
Permanent 404     : 0
Remaining this run: 87,461
Minimum request-time estimate: 182.2 minutes (di luar overhead/retry)



Checkpoint saved | processed this run=500 | records=500 | failures=0

Checkpoint saved | processed this run=1,000 | records=999 | failures=1

Checkpoint saved | processed this run=1,500 | records=1,499 | failures=1

Checkpoint saved | processed this run=2,000 | records=1,999 | failures=1

Checkpoint saved | processed this run=2,500 | records=2,499 | failures=1

Checkpoint saved | processed this run=3,000 | records=2,999 | failures=1

Checkpoint saved | processed this run=3,500 | records=3,499 | failures=1

Checkpoint saved | processed this run=4,000 | records=3,999 | failures=1

Checkpoint saved | processed this run=4,500 | records=4,497 | failures=3

Checkpoint saved | processed this run=5,000 | records=4,996 | failures=4

Checkpoint saved | processed this run=5,500 | records=5,496 | failures=4

Checkpoint saved | processed this run=6,000 | records=5,996 | failures=4

Checkpoint saved | processed this run=6,500 | records=6,496 | failures=4

Checkpoint saved | processed this run=7,000

In [12]:
# 11. Build final dataset

if TEST_MODE:
    print(
        "TEST_MODE=True. Final full dataset belum dibuat."
    )
else:
    final_df = (
        pd.DataFrame(records)
        .drop_duplicates(subset=["tmdbId"], keep="last")
        .sort_values("movieId")
        .reset_index(drop=True)
    )

    final_df.to_parquet(
        FINAL_PARQUET_PATH,
        index=False,
    )
    final_df.to_csv(
        FINAL_CSV_PATH,
        index=False,
    )

    print("Final shape:", final_df.shape)
    display(final_df.head())

Final shape: (86214, 30)


,movieId,tmdbId,imdbId,movielens_title,movielens_genres,tmdb_title,original_title,release_date,release_year,runtime,...,keyword_names_json,production_company_ids_json,production_company_names_json,collection_id,collection_name,n_directors,n_writers,n_top_cast,n_keywords,n_genres
0,1,862,114709,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Toy Story,1995-11-22,1995.0,81,...,"[""rescue"", ""friendship"", ""mission"", ""jealousy""...",[3],"[""Pixar""]",10194.0,Toy Story Collection,1,8,5,26,4
1,2,8844,113497,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Jumanji,1995-12-15,1995.0,104,...,"[""based on novel or book"", ""giant insect"", ""bo...","[559, 10201, 2550]","[""TriStar Pictures"", ""Interscope Communication...",495527.0,Jumanji Collection,1,4,5,11,3
2,3,15602,113228,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Grumpier Old Men,1995-12-22,1995.0,101,...,"[""fishing"", ""sequel"", ""old man"", ""best friend""...","[19464, 174]","[""Lancaster Gate"", ""Warner Bros. Pictures""]",119050.0,Grumpy Old Men Collection,1,1,5,9,2
3,4,31357,114885,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Waiting to Exhale,1995-12-22,1995.0,127,...,"[""based on novel or book"", ""single mother"", ""d...",[25],"[""20th Century Fox""]",NaN,None,1,2,5,11,3
4,5,11862,113041,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Father of the Bride Part II,1995-12-08,1995.0,106,...,"[""daughter"", ""baby"", ""parent child relationshi...","[9195, 5842]","[""Touchstone Pictures"", ""Sandollar Productions""]",96871.0,Father of the Bride (Steve Martin) Collection,1,4,5,14,2


In [13]:
# 12. Quality report

quality_df = test_df.copy() if TEST_MODE else final_df.copy()

def json_list_nonempty(series):
    return series.fillna("[]").map(
        lambda x: len(json.loads(x)) > 0
        if isinstance(x, str)
        else False
    )

quality_metrics = {
    "rows": int(len(quality_df)),
    "has_release_year": int(
        quality_df["release_year"].notna().sum()
    ),
    "has_runtime": int(
        quality_df["runtime"].notna().sum()
    ),
    "has_genres": int(
        json_list_nonempty(
            quality_df["genre_names_json"]
        ).sum()
    ),
    "has_director": int(
        json_list_nonempty(
            quality_df["director_names_json"]
        ).sum()
    ),
    "has_writer": int(
        json_list_nonempty(
            quality_df["writer_names_json"]
        ).sum()
    ),
    "has_top_cast": int(
        json_list_nonempty(
            quality_df["top_cast_names_json"]
        ).sum()
    ),
    "has_keywords": int(
        json_list_nonempty(
            quality_df["keyword_names_json"]
        ).sum()
    ),
    "has_production_company": int(
        json_list_nonempty(
            quality_df["production_company_names_json"]
        ).sum()
    ),
    "has_collection": int(
        quality_df["collection_id"].notna().sum()
    ),
}

report_rows = []
for field, count in quality_metrics.items():
    coverage = (
        100.0
        if field == "rows"
        else (
            100.0 * count / len(quality_df)
            if len(quality_df) > 0
            else 0.0
        )
    )
    report_rows.append({
        "field": field,
        "count": count,
        "coverage_pct": coverage,
    })

quality_report = pd.DataFrame(report_rows)
display(quality_report)

,field,count,coverage_pct
0,rows,86214,100.000000
1,has_release_year,86181,99.961723
2,has_runtime,86214,100.000000
3,has_genres,85567,99.249542
4,has_director,85733,99.442086
5,has_writer,77835,90.281161
6,has_top_cast,83139,96.433294
7,has_keywords,62830,72.876795
8,has_production_company,76236,88.426474
9,has_collection,9718,11.271951


In [14]:
# 13. Save summary + ZIP

if TEST_MODE:
    print(
        "TEST_MODE=True: output test belum dianggap dataset final."
    )
else:
    failed_df = (
        pd.DataFrame(failures)
        if failures
        else pd.DataFrame(
            columns=["movieId", "tmdbId", "error"]
        )
    )

    summary = {
        "project": "Aoranema TMDB Metadata Enrichment",
        "source": "TMDB API",
        "movielens_total_movies": int(total_movies),
        "movielens_valid_tmdb_ids": int(len(movie_map_valid)),
        "metadata_rows_collected": int(len(final_df)),
        "failed_requests": int(len(failed_df)),
        "coverage_pct_of_valid_tmdb_ids": float(
            100.0 * len(final_df) / len(movie_map_valid)
            if len(movie_map_valid) > 0
            else 0.0
        ),
        "top_cast_n": TOP_CAST_N,
        "max_writers": MAX_WRITERS,
        "language": LANGUAGE,
        "fields": list(final_df.columns),
        "quality": quality_metrics,
        "notes": {
            "tmdb_current_popularity_not_collected": True,
            "tmdb_current_vote_average_not_collected": True,
            "reason": (
                "Avoid current TMDB popularity/vote values "
                "as historical MovieLens training features."
            ),
        },
    }

    with open(
        SUMMARY_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2,
        )

    zip_path = shutil.make_archive(
        "/kaggle/working/aoranema_tmdb_metadata",
        "zip",
        OUTPUT_DIR,
    )

    print("Final Parquet:", FINAL_PARQUET_PATH)
    print("Final CSV    :", FINAL_CSV_PATH)
    print("Summary      :", SUMMARY_PATH)
    print("ZIP          :", zip_path)

    print("\nOutput files:")
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(
            "-",
            path.name,
            f"({path.stat().st_size / 1024 / 1024:.2f} MB)"
        )

Final Parquet: /kaggle/working/aoranema_tmdb_metadata/tmdb_metadata.parquet
Final CSV    : /kaggle/working/aoranema_tmdb_metadata/tmdb_metadata.csv
Summary      : /kaggle/working/aoranema_tmdb_metadata/metadata_info.json
ZIP          : /kaggle/working/aoranema_tmdb_metadata.zip

Output files:
- failed_tmdb_requests.csv (0.03 MB)
- metadata_info.json (0.00 MB)
- tmdb_metadata.csv (47.51 MB)
- tmdb_metadata.parquet (25.44 MB)
- tmdb_metadata_checkpoint.parquet (25.44 MB)


# Output untuk Recommendation V2

Setelah full collection selesai:

```text
aoranema_tmdb_metadata/
├── tmdb_metadata.parquet
├── tmdb_metadata.csv
├── tmdb_metadata_checkpoint.parquet
├── failed_tmdb_requests.csv
└── metadata_info.json
```

File utama untuk eksperimen V2:

`tmdb_metadata.parquet`

Nanti V2 menggunakan:

```text
MovieLens 32M
+
tmdb_metadata.parquet
```

untuk richer user preference, director/cast/writer/keyword affinity, per-user chronological split, negative sampling, dan ranking model.